# AutoML-M00: Índice Pre-Modelado AutoML

**TFM: Predicción de Abandono Universitario**

| | |
|---|---|
| **Autora** | María José Morte |
| **Email** | mjmorteruiz@uoc.edu (UOC) \| morte@uji.es (UJI) |

---

## 🎯 Objetivo

Generar página índice HTML para la fase Pre-Modelado AutoML.

## 📊 Dos casos de estudio

| Caso | Descripción | Features |
|------|-------------|----------|
| **D** | Alerta temprana (sin traidoras) | 21 |
| **D_strict** | Producción (más estricto) | 19 |

## 📝 Contexto

Los casos A y B originales presentaban **Target Leakage** (AUC=1.0).
Tras la auditoría (F3-M05→M08), se generaron los casos D y D_strict,
libres de fuga de información. Este AutoML trabaja exclusivamente
sobre datos limpios.

In [1]:
# ============================================================================
# CELDA 1: CONFIGURACIÓN
# ============================================================================

import sys, warnings
from pathlib import Path
warnings.filterwarnings('ignore')

if 'google.colab' in sys.modules:
    from google.colab import drive
    drive.mount('/content/drive')
    ROOT = Path('/content/drive/MyDrive/AU_UJI')
else:
    ROOT = Path.cwd()
    for _ in range(5):
        if (ROOT / 'src').exists(): break
        ROOT = ROOT.parent

sys.path.insert(0, str(ROOT))
import pandas as pd
from src.config import RUTA_AUTOML, RUTA_HTML, info_entorno
from src.utils import crear_directorios, formato_numero_es
from src.html import generar_kpis_html, generar_tarjetas_html, generar_seccion_html, generar_html_navegacion_completa, guardar_html
from src.html.render import render_base_html

RUTA_FASE_AUTOML_HTML = RUTA_HTML / 'fase_automl'
crear_directorios([RUTA_FASE_AUTOML_HTML])
fmt = formato_numero_es
info_entorno()

✓ Directorios verificados: 1
✓ ===========================================================================
✓ 📌 INFORMACIÓN DEL ENTORNO DEL PROYECTO
✓ ===========================================================================
✓ 🖥️  Entorno detectado: Local
✓ 📂 Ruta base:     C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI
✓ 📁 RAW:           C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\00_raw
✓ 📁 INTERIM:       C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\01_interim
✓ 📁 PROCESSED:     C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\02_processed
✓ 📁 FEATURES:      C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\03_features
✓ 📁 AUTOML:        C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\automl
✓ 📁 NOTEBOOKS:     C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\notebooks
✓ 📄 Excel principal: C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\00_raw\datos_proyecto_sin_preinscrip.xlsx
✓

In [2]:
# ============================================================================
# CELDA 2: CARGAR DATOS + GENERAR HTML
# ============================================================================

# Cargar dataset final (D_strict)
ruta_ds = RUTA_AUTOML / 'dataset_final_tfm.parquet'
ruta_d = RUTA_AUTOML / 'df_exp_automl_D.parquet'

if ruta_ds.exists():
    df_ds = pd.read_parquet(ruta_ds)
    n_reg = len(df_ds)
    n_feat_ds = df_ds.shape[1] - 1
    pct_ab = (df_ds['abandono']==1).mean()*100
    del df_ds
else:
    n_reg = 33621; n_feat_ds = 19; pct_ab = 29.2

if ruta_d.exists():
    df_d = pd.read_parquet(ruta_d)
    n_feat_d = df_d.shape[1] - 1
    del df_d
else:
    n_feat_d = 21

nav_fases, nav_modulos = generar_html_navegacion_completa(
    fase_activa='fase_automl', modulo_activo='indice')

KPIS = [
    {'valor': fmt(n_reg), 'titulo': 'Expedientes'},
    {'valor': str(n_feat_d), 'titulo': 'Features (D)'},
    {'valor': str(n_feat_ds), 'titulo': 'Features (D_strict)'},
    {'valor': f'{pct_ab:.1f}%', 'titulo': 'Abandono'},
]

MODULOS = [
    {'archivo': 'm01_baselines.html', 'emoji': '📊', 'titulo': 'M01: Baselines',
     'desc': 'LogReg, RF, GBM, XGBoost — referencia base.', 'color': '#3182ce'},
    {'archivo': 'm02_lazypredict.html', 'emoji': '⚡', 'titulo': 'M02: LazyPredict',
     'desc': 'Screening ~30 modelos sin tuning.', 'color': '#d69e2e'},
    {'archivo': 'm03_pycaret.html', 'emoji': '🤖', 'titulo': 'M03: PyCaret',
     'desc': 'AutoML con compare, tune, blend.', 'color': '#38a169'},
    {'archivo': 'm04_h2o.html', 'emoji': '💧', 'titulo': 'M04: H2O',
     'desc': 'AutoML distribuido + stacking.', 'color': '#805ad5'},
    {'archivo': 'm05_autogluon.html', 'emoji': '🚀', 'titulo': 'M05: AutoGluon',
     'desc': 'Ensemble SOTA con multi-layer stacking.', 'color': '#ed8936'},
    {'archivo': 'm06_comparativa.html', 'emoji': '🏆', 'titulo': 'M06: Comparativa',
     'desc': 'Ranking final D vs D_strict + Gráfico de la Verdad.', 'color': '#e53e3e'},
]

tarjetas = [{'titulo': m['titulo'], 'descripcion': m['desc'], 'emoji': m['emoji'],
             'link': m['archivo'], 'link_texto': 'Abrir →', 'color': m['color']} for m in MODULOS]

s1 = generar_seccion_html('Resumen', f'''
<p>Exploración de <strong>4 frameworks AutoML</strong> sobre <strong>{fmt(n_reg)} expedientes</strong>
con <strong>2 casos limpios</strong> (D: {n_feat_d} features, D_strict: {n_feat_ds} features).</p>
<div style="background:#F0FFF4;padding:15px;border-radius:8px;margin-top:15px;border-left:4px solid #38a169;">
  <strong>✅ Datos auditados:</strong> Sin leakage, sin constantes, sin traidoras.
  Los casos A y B (con fuga) están archivados en <code>old/</code> como control.
</div>
<div style="background:#EBF8FF;padding:15px;border-radius:8px;margin-top:10px;border-left:4px solid #3182ce;">
  <strong>📌 Entornos conda:</strong> Cada framework se ejecuta en su propio entorno.
</div>''', '⚡')

html = render_base_html(
    titulo='⚡ Pre-Modelado: AutoML', icono='⚡',
    subtitulo=f'Exploración de Frameworks AutoML (Caso D + D_strict)',
    nav_fases=nav_fases, nav_modulos=nav_modulos,
    contenido=s1 + generar_kpis_html(KPIS) + generar_seccion_html('Módulos', generar_tarjetas_html(tarjetas), '📦'),
    notebook_nombre='fautoml_m00_indice.ipynb', notebook_carpeta='fase_automl')
ruta_html = RUTA_FASE_AUTOML_HTML / 'fase_automl_index.html'
guardar_html(html, ruta_html)
print(f'HTML: {ruta_html}')

✅ HTML guardado: C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\docs\html\fase_automl\fase_automl_index.html
HTML: C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\docs\html\fase_automl\fase_automl_index.html


In [3]:
# ============================================================================
# CELDA 3: RESUMEN
# ============================================================================

print('\n' + '='*60)
print('✅ AUTOML-M00 COMPLETADO')
print('='*60)
print(f'Archivo: {ruta_html}')
print(f'Expedientes: {fmt(n_reg)}')
print(f'Casos: D ({n_feat_d} features) + D_strict ({n_feat_ds} features)')
print(f'Target: {pct_ab:.1f}% abandono')
print(f'\n🎯 Siguiente: fautoml_m01_baselines.ipynb')
print('='*60)


✅ AUTOML-M00 COMPLETADO
Archivo: C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\docs\html\fase_automl\fase_automl_index.html
Expedientes: 33.621
Casos: D (21 features) + D_strict (19 features)
Target: 29.2% abandono

🎯 Siguiente: fautoml_m01_baselines.ipynb
